깃허브 연동

In [1]:
!git clone https://github.com/hman930/hman930.git

Cloning into 'hman930'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 2.97 MiB | 13.17 MiB/s, done.
Resolving deltas: 100% (1/1), done.


구글 드라이브 마운트-파일 업로드

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import pandas as pd

file_path = '/content/drive/MyDrive/(test)부산광역시_유저데이터.csv'
df = pd.read_csv(file_path)
df.head()

,부모닉네임,아이나이,성별,성격,거주지역,선호활동
0,봄봄맘,2,남,적극적,부산-중구,블록쌓기
1,한별맘,6,여,적극적,부산-중구,"음악듣기, 산책"
2,행복이맘,0,남,사교적,부산-중구,"놀이, 퍼즐맞추기"
3,소윤맘,2,여,사교적,부산-중구,"놀이, 미술놀이"
4,은율맘,1,여,수줍음,부산-중구,미술놀이


Faiss 기반 추천 시스템

In [15]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 39.9 MB/s eta 0:00:00


데이터 전처리 및 벡터화

In [20]:
df['선호활동'].head()

,선호활동
0,블록쌓기
1,"음악듣기, 산책"
2,"놀이, 퍼즐맞추기"
3,"놀이, 미술놀이"
4,미술놀이


In [21]:
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
import numpy as np

# 범주형 인코딩
categorical_cols = ['성별', '성격', '거주지역']
encoder = OneHotEncoder(sparse_output=False)
encoded_cat = encoder.fit_transform(df[categorical_cols])

# 선호활동 다중 선택 인코딩
mlb = MultiLabelBinarizer()
encoded_activities = mlb.fit_transform(df['선호활동'])

# 나이(수치)
age = df[['아이나이']].values

# 사용자 벡터 생성
user_vectors = np.hstack([age, encoded_cat, encoded_activities])

#구 단위 지역 좌표 기반 + 사용자 특성 기반 추천 알고리즘

시-구 단위로 지역 좌표 기반으로 우선 순위 추천 후
사용자 특성 기반을 바탕으로 추천 알고리즘 형성

In [22]:
import numpy as np
import faiss

# user_vectors는 이전 단계에서 만든 벡터
user_vectors = user_vectors.astype('float32')

In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from math import radians, sin, cos, sqrt, atan2

# 📌 1. 지역별 좌표 정의
region_coords = {
    "부산-중구": (35.1065, 129.0326),
    "부산-서구": (35.0961, 129.0233),
    "부산-동구": (35.1293, 129.0450),
    "부산-영도구": (35.0913, 129.0676),
    "부산-부산진구": (35.1629, 129.0556),
    "부산-동래구": (35.2054, 129.0836),
    "부산-남구": (35.1368, 129.0842),
    "부산-북구": (35.1976, 128.9902),
    "부산-해운대구": (35.1631, 129.1636),
    "부산-사하구": (35.1044, 128.9745),
    "부산-금정구": (35.2435, 129.0924),
    "부산-강서구": (35.2122, 128.9802),
    "부산-연제구": (35.1842, 129.0790),
    "부산-수영구": (35.1424, 129.1131),
    "부산-사상구": (35.1543, 128.9905),
    "부산-기장군": (35.2447, 129.2224)
}

# 📌 2. Haversine 거리 계산 함수
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# 📌 3. 거리 기반 점수 함수
def distance_score(km):
    if km <= 5:
        return 1.0
    elif km <= 10:
        return 0.8
    elif km <= 15:
        return 0.6
    else:
        return 0.3

In [30]:
# 📌 4. 데이터 준비 (df 변수는 기존과 동일하게 사용)
df["선호활동"] = df["선호활동"].apply(lambda x: x.split(", ") if isinstance(x, str) else x)
df["위도"] = df["거주지역"].apply(lambda x: region_coords[x][0])
df["경도"] = df["거주지역"].apply(lambda x: region_coords[x][1])

# 타겟 사용자 설정
target_nickname = "하람맘"
target_index = df[df["부모닉네임"] == target_nickname].index[0]
target_lat = df.loc[target_index, "위도"]
target_lon = df.loc[target_index, "경도"]

# 거리 및 지역 유사도 점수 계산
df["거리_km"] = df.apply(
    lambda row: haversine_distance(target_lat, target_lon, row["위도"], row["경도"]),
    axis=1
)
df["지역유사도점수"] = df["거리_km"].apply(distance_score)

# 사용자 특성 벡터화
encoder = OneHotEncoder(sparse_output=False)
encoded_cat = encoder.fit_transform(df[["성별", "성격"]])

mlb = MultiLabelBinarizer()
encoded_activities = mlb.fit_transform(df["선호활동"])

age = df[["아이나이"]].values
user_vectors = np.hstack([age, encoded_cat, encoded_activities]).astype("float32")

# 코사인 유사도 계산
feature_sim = cosine_similarity(user_vectors)[target_index]

# 최종 추천 점수 계산 (지역 50% + 특성 50%)
df["유사도"] = feature_sim.round(3)
df["추천점수"] = df["지역유사도점수"] * 0.5 + feature_sim * 0.5

# 자기 자신 제외하고 상위 추천 10명 출력
recommended_df = df.drop(index=target_index).sort_values(by="추천점수", ascending=False).head(10).copy()

print(recommended_df[["부모닉네임","거주지역", "아이나이", "선호활동", "추천점수"]])

    부모닉네임    거주지역  아이나이               선호활동      추천점수
195   별이네  부산-영도구     5              [책읽기]  0.982143
40    아리맘   부산-중구     7          [책읽기, 산책]  0.980236
119   온유맘   부산-동구     4             [음악듣기]  0.976911
79    다솜맘   부산-서구     7    [미술놀이, 대화, 책읽기]  0.975769
42    아인맘   부산-중구     7    [책읽기, 요리놀이, 산책]  0.975769
45   은하수맘   부산-중구     7      [산책, 대화, 책읽기]  0.975769
127   예서맘   부산-동구     6  [블록쌓기, 미술놀이, 책읽기]  0.972225
114   노을맘   부산-동구     6    [책읽기, 블록쌓기, 산책]  0.972225
7     로아맘   부산-중구     7               [놀이]  0.971728
10    루다맘   부산-중구     7               [놀이]  0.971728


In [32]:
!ls hman930

crawling_notion.ipynb  README.md
